# Negative Binomial Model for Cyclist Volume in Adelaide

## Variables:
- POW_CTW: N people who work in this DZN and cycle to work
    - ```all-data/census-formatted/greater-adelaide/MTW-Bicycle-POW-DZN.csv``` [Bicycle]
- POW_FE: N people who work in this DZN and are fullly engaged in employment or study
    - ```all-data/census-formatted/greater-adelaide/EETP-POW-DZN.csv``` [Fully Engaged]
- INF_CW: 1 if the edge the count point lies on is a cycleway, else 0
- MB population density: PD_MB
- Distance to nearest large DZN: DNL_DZN
- Age demographics, try males 30-40 and females 15-25 (best from Dadshova's): M_30_40, F_15_25
- Households with a THI of > 3500, place of enumeration, MB: H_3500
- Manual Overpass IS_CYCLE_ROUTE: INF_CR

__Monash Bicycle Infrastructure Classification__ - Binary dummy variables for every type of bicycle infrastructure

- MBI_BP: Dedicated bike path, Shared bike path
- MBI_PP: Pedestrian path/street with cycling allowed
- MBI_PL: Protected bike lane
- MBI_BL: Buffered bike lane (kerb-side), Buffered bike lane (road-side), Buffered bike lane (both sides)
- MBI_PTL: Painted bike lane, Painted bike lane (single-side)
- MBI_Ad: Advisory bike lane, Advisory bike lane (single-side), Peak hour advisory bike lane, Peak hour advisory bike lane (single-side)
- MBI_PkPt: Peak hour painted bike lane, Peak hour painted bike lane (single-side)
- MBI_Sh: Sharrow
- MBI_SZ: Shared zone
- MBI_Bus: Bus lane with cycling allowed
- MBI_Mx: Mixed traffic
- MBI_No: Bicyclists dismount, Pedestrian path/street with cycling not allowed

## Correlation Risks:

Probably all the Monash Classifiers

__*Generate Adelaide Graph Network*__

In [1]:
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import geopandas as gpd

ox.settings.log_console = True
ox.settings.use_cache = True
ox.settings.useful_tags_way += [
    "cycleway", "cycleway:left", "cycleway:right", "cycleway:both",
    "cycleway:left:lane", "cycleway:right:lane", "cycleway:both:lane",
    "cycleway:left:buffer:left", "cycleway:left:buffer:right", "cycleway:left:buffer:both",
    "cycleway:right:buffer:left", "cycleway:right:buffer:right", "cycleway:right:buffer:both",
    "cycleway:both:buffer:left", "cycleway:both:buffer:right", "cycleway:both:buffer:both",
    "cycleway:left:conditional", "cycleway:right:conditional", "cycleway:both:conditional",
    "cycleway:left:lane:conditional", "cycleway:right:lane:conditional", "cycleway:both:lane:conditional",
    "foot", "bicycle", "route", "name"
]

# Monkey-patch deprecated aliases so old code doesn't crash
import osmnx.settings as ox_settings
if not hasattr(ox_settings, 'http_user_agent'):
    ox_settings.http_user_agent = ox_settings.default_user_agent
if not hasattr(ox_settings, 'http_referer'):
    ox_settings.http_referer = ox_settings.default_referer
if not hasattr(ox_settings, 'nominatim_url'):
    ox_settings.nominatim_url = ox_settings.nominatim_endpoint
if not hasattr(ox_settings, 'http_accept_language'):
    ox_settings.http_accept_language = ox_settings.default_accept_language

In [2]:
gdf_boundary = ox.geocode_to_gdf("R11381689", by_osmid=True)
polygon = gdf_boundary.geometry.iloc[0]

# Monash dual-filter approach to capture all relevant OSM tags
bike_filter_1 = '[~"highway"~"cycleway|trunk|primary|secondary|tertiary|unclassified|residential|primary_link|secondary_link|tertiary_link|living_street|trailhead|service"]["access"!~"no|private"]["bicycle"!~"no"]["area"!~"yes"]'
G1 = ox.graph_from_polygon(polygon, custom_filter=bike_filter_1, simplify=False, retain_all=True)

bike_filter_2 = '[~"highway"~"footway|pedestrian|path"]["bicycle"~"yes|designated|dismount"]["area"!~"yes"]'
G2 = ox.graph_from_polygon(polygon, custom_filter=bike_filter_2, simplify=False, retain_all=True)

G = nx.compose(G1, G2)

In [3]:
# Debug: Inspect edge attributes loaded from OSM
import pandas as pd

nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)

print(f"=== Graph Stats ===")
print(f"Nodes: {len(nodes)}")
print(f"Edges: {len(edges)}")
print(f"Edge columns ({len(edges.columns)}): {list(edges.columns)}")

print(f"\n=== Sample of cycleway-related tags ===")
cycleway_cols = [c for c in edges.columns if 'cycleway' in c.lower() or c in ['highway', 'foot', 'bicycle', 'name', 'route']]
print(f"Relevant columns: {cycleway_cols}")

for col in cycleway_cols:
    if col in edges.columns:
        non_null = edges[col].notna().sum()
        if non_null > 0:
            print(f"\n{col}: {non_null} non-null values")
            print(edges[col].value_counts().head(10))

print(f"\n=== First 5 edges ===")
print(edges[cycleway_cols].head())

# Check for key classification tags
key_tags = [
    'cycleway:left:buffer:left', 'cycleway:left:buffer:right', 'cycleway:left:buffer:both',
    'cycleway:both:buffer:left', 'cycleway:both:buffer:right', 'cycleway:both:buffer:both',
    'cycleway:left:lane', 'cycleway:both:lane',
    'cycleway:left:conditional', 'cycleway:left:lane:conditional',
    'cycleway:both:conditional', 'cycleway:both:lane:conditional',
    'highway', 'foot', 'bicycle', 'name', 'route'
]
print(f"\n=== Key classification tag coverage ===")
for tag in key_tags:
    if tag in edges.columns:
        nn = edges[tag].notna().sum()
        print(f"  {tag}: {nn} non-null")
    else:
        print(f"  {tag}: MISSING")

=== Graph Stats ===
Nodes: 466234
Edges: 893280
Edge columns (56): ['osmid', 'surface', 'highway', 'cycleway:left', 'junction', 'oneway', 'reversed', 'length', 'name', 'maxspeed', 'lanes', 'ref', 'foot', 'cycleway:both', 'bridge', 'cycleway', 'cycleway:left:lane', 'cycleway:right', 'width', 'cycleway:both:lane', 'bicycle', 'smoothness', 'parking:both', 'oneway:bicycle', 'motorcar', 'motor_vehicle', 'cycleway:left:oneway', 'segregated', 'parking:left', 'parking:right', 'service', 'cycleway:right:lane', 'access', 'parking:lane:left', 'parking:lane:right', 'parking:lane:both', 'cycleway:right:oneway', 'traffic_calming', 'cycleway:surface', 'route', 'mtb', 'vehicle', 'tunnel', 'mtb:scale', 'mtb:scale:imba', 'cycleway:left:segregated', 'tracktype', 'cycleway:both:segregated', 'cycleway:left:conditional', 'cycleway:right:conditional', 'cycleway:left:width', 'cycleway:right:width', 'cycleway:width', 'landuse', 'path', 'geometry']

=== Sample of cycleway-related tags ===
Relevant columns: ['hi

In [ ]:
# test overpass is responding before getting cycle routes


__*Add in cycle route relations*__

In [4]:
import sys
sys.path.insert(0, '/Users/alexhuynh/Documents/AdelaideCyclistVolumesModel')
from infrastructure.cycle_routes import enrich_graph_with_cycle_routes
# Get the graph with IS_CYCLE_ROUTE edge attribute
G = enrich_graph_with_cycle_routes(G)

  Sub-query 1/4: found 629 ways
  Sub-query 2/4 error: OverpassUnknownHTTPStatusCode: Unknown/Unhandled status code: 502 — retrying in 1s...
  Sub-query 2/4: found 58 ways
  Sub-query 3/4 error: OverpassUnknownHTTPStatusCode: Unknown/Unhandled status code: 502 — retrying in 1s...
  Sub-query 3/4: found 2085 ways
  Sub-query 4/4 error: OverpassUnknownHTTPStatusCode: Unknown/Unhandled status code: 502 — retrying in 1s...
  Sub-query 4/4: found 1154 ways
Total ways found across 4 sub-queries: 2594
Found 2594 ways in bicycle routes


__*Import Street Network Data*__

In [5]:
from infrastructure.get_bike_infra import enrich_graph_with_monash_classifiers
G = enrich_graph_with_monash_classifiers(G)

In [7]:
from itertools import islice

for node_data in islice(G.nodes(data=True), 10):
    print(node_data)

(26528915, {'y': -34.8111463, 'x': 138.6110184, 'street_count': 2})
(1874354990, {'y': -34.8111436, 'x': 138.6109949, 'street_count': 2})
(1850962866, {'y': -34.8111366, 'x': 138.6109713, 'street_count': 2})
(26547862, {'y': -34.8121827, 'x': 138.6114304, 'street_count': 2})
(13148430796, {'y': -34.8121918, 'x': 138.611935, 'street_count': 2, 'highway': 'crossing'})
(13148432306, {'y': -34.8121812, 'x': 138.6111099, 'street_count': 2, 'highway': 'crossing'})
(263714021, {'y': -34.8121941, 'x': 138.6120619, 'street_count': 3})
(26547873, {'y': -34.812181, 'x': 138.6110713, 'street_count': 2})
(1874355250, {'y': -34.8121817, 'x': 138.6109961, 'street_count': 3})
(1874355237, {'y': -34.8120143, 'x': 138.6109986, 'street_count': 2})


__*Load MBs and DZNs into graph edges*__

In [8]:
mbs_aus = gpd.read_file("../all-data/australia-mesh-blocks-raw/MB_2021_AUST_GDA2020.shp")
dzns_aus = gpd.read_file("../all-data/australia-destination-zones-raw/DZN_2021_AUST_GDA2020.shp")

mb_gdf_boundary_crs = gdf_boundary.to_crs(mbs_aus.crs)
dzn_gdf_boundary_crs = gdf_boundary.to_crs(dzns_aus.crs)
mbs = gpd.clip(mbs_aus, mb_gdf_boundary_crs)
dzns = gpd.clip(dzns_aus, dzn_gdf_boundary_crs)

print('MB all count: ', len(mbs_aus))
print('MB Adelaide count: ', len(mbs))
print('DZN all count: ', len(dzns_aus))
print('DZN Adelaide count: ', len(dzns))

MB all count:  368286
MB Adelaide count:  18263
DZN all count:  9329
DZN Adelaide count:  479


In [9]:
nodes_gdf = ox.graph_to_gdfs(G, nodes=True, edges=False)[["geometry"]]
 
nodes_mbs = gpd.sjoin(
    nodes_gdf.to_crs(mbs.crs),
    mbs[["MB_CODE21", "geometry"]],
    how="left",
    predicate="within",
)
node_mb_codes = nodes_mbs["MB_CODE21"].dropna().to_dict()
nx.set_node_attributes(G, node_mb_codes, "MB_CODE21")

nodes_dzns = gpd.sjoin(
    nodes_gdf.to_crs(dzns.crs),
    dzns[["DZN_CODE21", "geometry"]],
    how="left",
    predicate="within",
)
node_dzn_codes = nodes_dzns["DZN_CODE21"].dropna().to_dict()
nx.set_node_attributes(G, node_dzn_codes, "DZN_CODE21")

print(f"Graph nodes: {len(G.nodes)}")
print(f"Nodes with MB_CODE21: {len(node_mb_codes)}")
print(f"Nodes with DZN_CODE21: {len(node_dzn_codes)}")

Graph nodes: 466234
Nodes with MB_CODE21: 466150
Nodes with DZN_CODE21: 466150


In [10]:
import pandas as pd
nodes_to_remove = [
    node_id
    for node_id, node_data in G.nodes(data=True)
    if not (
        pd.notna(node_data.get("MB_CODE21"))
        or pd.notna(node_data.get("DZN_CODE21"))
    )
]

G.remove_nodes_from(nodes_to_remove)
print(f"Removed {len(nodes_to_remove)} nodes; remaining nodes: {len(G.nodes)}")
print(f"Remaining edges: {len(G.edges)}")

Removed 84 nodes; remaining nodes: 466150
Remaining edges: 893094


## Retrieve following vars for graph and regression table

__Generic Census Data:__
- DNL_DZN
- PD_MB

__Census Table Data:__
- POW_CTW
- POW_FE
- M_30_40
- F_15_25
- H_3500

__Infrastructure Data:__
- INF_LRG_RD
- INF_MED_RD
- INF_SML_RD
- INF_CL
- INF_CR
- INF_CW

__*DNL_DZN*__

Distance to nearest large destination zone (8000+ jobs). These zones include the areas near North Terrace / Rundle / Franklin st, the RAH, between Pirie and Wakefield St, East End. 8000 is an arbitrary number.